In [ ]:
import csv

id_to_categories = {}
rename_ids = []
with open("/scr/BEHAVIOR-1K/asset_pipeline/metadata/object_renames.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        from_category = row["Original category (auto)"]
        to_category = row["New Category"]
        id_to_categories[row["ID (auto)"]] = (from_category, to_category)
        rename_ids.append(row["ID (auto)"])

In [ ]:
deletion_ids = set()
with open("/scr/BEHAVIOR-1K/asset_pipeline/metadata/deletion_queue.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row["Object"]
        deletion_ids.add(name.split("-")[1])
deletion_ids

In [ ]:
import json, pathlib

inventory = json.loads(
    pathlib.Path(
        "/scr/BEHAVIOR-1K/asset_pipeline/artifacts/pipeline/object_inventory.json"
    ).read_text()
)
provided_objects = set(inventory["providers"].keys())
needed_objects = set(inventory["needed_by"].keys())
all_objects = sorted(provided_objects | needed_objects)

In [ ]:
with open("/scr/BEHAVIOR-1K/asset_pipeline/metadata/category_mapping.csv", "r") as f:
    reader = csv.DictReader(f)
    valid_categories = {row["category"] for row in reader}

In [ ]:
real_cats_by_id = {}
for obj in all_objects:
    cat, mdl = obj.split("-")
    real_cats_by_id[mdl] = cat
    if mdl in id_to_categories:
        from_cat, to_cat = id_to_categories[mdl]
        link = f"https://behavior.stanford.edu/knowledgebase/objects/{cat}-{mdl}/index.html"
        from_val = "valid" if from_cat in valid_categories else "invalid"
        to_val = "valid" if to_cat in valid_categories else "invalid"
        cat_val = "valid" if cat in valid_categories else "invalid"
        if cat != from_cat and cat != to_cat:
            print(
                f"{link}: expected category to go from {from_cat}({from_val}) to {to_cat}({to_val}), found {cat} ({cat_val})"
            )

for obj in deletion_ids:
    if obj in real_cats_by_id:
        del real_cats_by_id[obj]

In [ ]:
# Find all rename entries that are not in the inventory
missing = set(rename_ids) - set(real_cats_by_id.keys())
missing

In [ ]:
for id in rename_ids:
    from_cat, to_cat = id_to_categories[id]
    print(int(id not in real_cats_by_id or real_cats_by_id[id] == to_cat))

In [ ]:
import collections

d = collections.defaultdict(set)
for obj in all_objects:
    cat, mdl = obj.split("-")
    d[mdl].add(obj)

d = {k: v for k, v in d.items() if len(v) > 1}
d